In [ ]:
# Install the Kaggle API client
!pip install kaggle

In [ ]:
import os
import json

# Your Kaggle username
KAGGLE_USERNAME = 'mariamkapanadze'
# Your Kaggle API key (provided in previous context)
KAGGLE_API_KEY = 'KGAT_50dfc0cd14d716963419e195a5dce230'

# Create the .kaggle directory if it doesn't exist
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True);

# Create kaggle.json file
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_API_KEY}

with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_config, f)

# Set permissions for the kaggle.json file
os.chmod(kaggle_json_path, 0o600)

print("Kaggle API key configured successfully!")

In [ ]:
# Download data for the 'Challenges in Representation Learning: Facial Expression Recognition Challenge'
!kaggle competitions download -c 'challenges-in-representation-learning-facial-expression-recognition-challenge'

# You might also want to unzip the downloaded files
# import zipfile
# with zipfile.ZipFile('challenges-in-representation-learning-facial-expression-recognition-challenge.zip', 'r') as zip_ref:
#     zip_ref.extractall('./fer2013') # Extract to a new directory named fer2013
!cat ~/.kaggle/kaggle.json

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.competition_download(
    "challenges-in-representation-learning-facial-expression-recognition-challenge"
)

print(path)

In [ ]:
import os

path = "/root/.cache/kagglehub/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge"

print(os.listdir(path))

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

path = "/root/.cache/kagglehub/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge"
train_df = pd.read_csv(os.path.join(path, 'train.csv'))
class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pixels = np.fromstring(row['pixels'], dtype=np.uint8, sep=' ')
        image = pixels.reshape(48, 48)

        label = int(row['emotion'])

        if self.transform:
            image = self.transform(image)

        return image, label
base_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = FERDataset(train_df, transform=base_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)

დავაკავშიროთ WandB

In [ ]:
!pip install wandb

In [ ]:
import wandb; wandb.login()

In [ ]:
wandb.init(
    project="facial-expression-recognition",
    name="baseline_small_cnn",
    config={
        "learning_rate": 0.001,
        "architecture": "Small-CNN",
        "dataset": "FER2013",
        "epochs": 10,
        "batch_size": 64
    }
)

Sanity Check here. შევამოწმოთ Single Batch Overfitting სტრატეგია ძალიან მარტივ ქსელზე, რათა დავამტკიცოთ, რომ კოდში ბაგები არ გვაქვს.

In [ ]:
import torch.nn as nn
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 48x48 -> 24x24
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 24x24 -> 12x12
        )
        self.classifier = nn.Linear(32 * 12 * 12, 7) # 7 კლასი (ემოცია)

    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2), # 24x24

            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 12x12
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

In [ ]:
class OptCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2), # 24x24
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), # 12x12
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), # 6x6
            nn.Dropout(0.4)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 256),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0), -1))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_experiment(model_class, model_name, exp_name, lr=0.001, batch_size=64, weight_decay=0.0):
    #ინიციალიზაცია
    wandb.init(
        project="facial-expression-recognition",
        name=f"{model_name}_{exp_name}",
        config={
            "model_architecture": model_name,
            "experiment_type": exp_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "weight_decay": weight_decay,
            "epochs": 10
        }
    )

    # DataLoader-ის ხელახალი შექმნა
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    epochs = 10
    print(f"\n starting: {model_name} | experiment: {exp_name}")
    print(f"Configs: LR={lr}, Batch Size={batch_size}, Weight Decay={weight_decay}")

    for epoch in range(epochs):
        # trainind
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_set)
        train_acc = 100.0 * correct / total

        # validaciis nawili
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        valid_loss = val_loss / len(val_set)
        val_acc = 100.0 * val_correct / val_total

        print(f"Epoch {epoch+1:02d} -> Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {valid_loss:.4f}, Val Acc: {val_acc:.2f}%")

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": valid_loss,
            "val_acc": val_acc
        })

    wandb.finish()
    print(f"✅ {model_name}_{exp_name} finished and logged\n")

experiment 1: baseline

In [ ]:
run_experiment(SmallCNN, "SmallCNN", "Baseline", lr=0.001, batch_size=64)

experiment 1.2: high learning rate underfittingis სანახავად

In [ ]:
run_experiment(SmallCNN, "SmallCNN", "High_LR", lr=0.01, batch_size=64)

experiment 1.3 : small batch + regularization

In [ ]:
run_experiment(SmallCNN, "SmallCNN", "Small_Batch_WD", lr=0.001, batch_size=32, weight_decay=1e-4)

experiment 2.1: baseline overfittingს ვხედავთ

In [ ]:
run_experiment(DeepCNN, "DeepCNN", "Baseline", lr=0.001, batch_size=64)

experiment

In [ ]:

run_experiment(DeepCNN, "DeepCNN", "High_LR", lr=0.01, batch_size=64)


In [ ]:
run_experiment(DeepCNN, "DeepCNN", "Small_Batch_WD", lr=0.001, batch_size=32, weight_decay=1e-4)

Experiment 3.1: baseline

In [ ]:
run_experiment(OptCNN, "OptimalCNN", "Baseline", lr=0.001, batch_size=64)


experiment 3.2: opt cnn with high learning rate

In [ ]:
run_experiment(OptCNN, "OptimalCNN", "High_LR", lr=0.01, batch_size=64)

experiment3.3 : opt snn with small batch and regularization

In [ ]:
run_experiment(OptCNN, "OptimalCNN", "Small_Batch_WD", lr=0.001, batch_size=32, weight_decay=1e-4)

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

class AdvancedFERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pixels = np.fromstring(row['pixels'], dtype=np.uint8, sep=' ')
        image = pixels.reshape(48, 48)
        label = int(row['emotion'])
        if self.transform: image = self.transform(image)
        return image, label


indices = list(range(len(train_df)))
split = int(0.8 * len(train_df))
train_indices, val_indices = indices[:split], indices[split:]

train_set = AdvancedFERDataset(train_df.iloc[train_indices], transform=train_transform)
val_set = AdvancedFERDataset(train_df.iloc[val_indices], transform=val_transform)

print("this is ready")

In [ ]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class FaceResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_channels = 32

        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )


        self.layer1 = ResidualBlock(32, 64, stride=2)  # 48x48 -> 24x24
        self.layer2 = ResidualBlock(64, 128, stride=2) # 24x24 -> 12x12
        self.layer3 = ResidualBlock(128, 256, stride=2) # 12x12 -> 6x6

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        out = self.init_conv(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = self.classifier(out)
        return out

print("model ready")

In [ ]:
def run_advanced_experiment(model_class, model_name, exp_name, epochs=30, lr=0.001, batch_size=64):
    wandb.init(
        project="facial-expression-recognition",
        name=f"{model_name}_{exp_name}",
        config={
            "model_architecture": model_name,
            "experiment_type": exp_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "epochs": epochs,
            "augmented_data": True
        }
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    print(f"\n start training : {model_name} ({epochs} epoch")

    for epoch in range(epochs):
        # treining
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_set)
        train_acc = 100.0 * correct / total

        # validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        valid_loss = val_loss / len(val_set)
        val_acc = 100.0 * val_correct / val_total

        # validation
        scheduler.step(valid_loss)

        print(f"Epoch {epoch+1:02d}/{epochs} -> Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {valid_loss:.4f}, Val Acc: {val_acc:.2f}%")

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": valid_loss,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]['lr']
        })

    wandb.finish()
    print(f" {model_name}_{exp_name} finished and logged")

In [ ]:
run_advanced_experiment(FaceResNet, "FaceResNet", "Augmented_30Epochs", epochs=30, lr=0.001, batch_size=64)